# Profiling Python Code
## Motivation

When optimizing scientific or high-performance Python code, intuition is usually wrong.

Many developers optimize:
- the wrong function,
- the wrong loop,
- or code that contributes almost nothing to total runtime.

Profiling answers a fundamental question:

> "Where is the program actually spending time?"

Optimization without profiling is often wasted effort.

The standard workflow is:

1. Measure runtime naively
2. Benchmark more rigorously
3. Profile function-level hotspots
4. Profile line-by-line bottlenecks
5. Analyze memory behavior
6. Optimize only the dominant costs

The standard Python libraries that we can use to profile are:

In [ ]:
import time
import timeit
import cProfile
import pstats

## Example Problem

We begin with a deliberately inefficient implementation.

In [ ]:
def slow_sum_of_squares(n):
    total = 0

    for i in range(n):
        total += i ** 2

    return total

## 1. Naive Timing with `time`

The simplest timing strategy uses wall-clock measurements.

This gives a first estimate of runtime, but single measurements are noisy.

In [ ]:
for i in range(5):   
    start = time.perf_counter()
    slow_sum_of_squares(10_000_000)
    end = time.perf_counter()
    print(f"Elapsed time run {i}: {end - start:.6f} seconds")

### Why `perf_counter()`?

Python provides multiple clocks.

`time.perf_counter()` is preferred because:
- high precision,
- includes sleep time,
- designed for benchmarking.

Avoid:
- `time.time()`
- manual stopwatch approaches

### Problem with Naive Timing

Single-run measurements are noisy.

Runtime may vary due to:
- CPU frequency scaling,
- cache effects,
- OS scheduling,
- background processes,
- JIT warmups.

We need repeated measurements.

## 2. Better Benchmarking with `timeit`

`timeit` executes code multiple times and reports more reliable timing statistics.

This is the preferred approach for:
- kernels,
- numerical loops,
- micro-optimizations.

In [ ]:
execution_time = timeit.timeit(
    "slow_sum_of_squares(10_000_000)",
    globals=globals(),
    number=5
)

print(f"Average execution time: {execution_time / 5:.6f} seconds")

In [ ]:
%timeit slow_sum_of_squares(10_000_000)

## Timing Multiple Variants

Suppose we compare:
- pure Python,
- NumPy vectorization.

In [ ]:
import numpy as np
def python_sum_of_squares(n):
    total = 0

    for i in range(n):
        total += i ** 2

    return total


def numpy_sum_of_squares(n):
    x = np.arange(n)
    return np.sum(x ** 2)

In [ ]:
python_time = timeit.timeit(
    "python_sum_of_squares(10_000_000)",
    globals=globals(),
    number=3
)

numpy_time = timeit.timeit(
    "numpy_sum_of_squares(10_000_000)",
    globals=globals(),
    number=3
)

print(f"Python average : {python_time / 3:.6f} s")
print(f"NumPy average  : {numpy_time / 3:.6f} s")

In [ ]:
%timeit python_sum_of_squares(10_000_000)
%timeit numpy_sum_of_squares(10_000_000)

NumPy is typically much faster because:
- loops execute in optimized C,
- vectorized operations reduce Python interpreter overhead,
- memory access patterns are optimized.

This demonstrates a central HPC principle:

> Python itself is often not slow.
> Python loops are slow.

## 3. Function-Level Profiling with `cProfile`

Timing tells us *how long* something takes.

Profiling tells us:
- which functions dominate runtime,
- how often they are called,
- cumulative execution costs.

In [ ]:
def expensive_operation():
    total = 0

    for i in range(1_000_000):
        total += np.sqrt(i)

    return total


def main():
    for _ in range(5):
        expensive_operation()

In [ ]:
cProfile.run("main()")

### Understanding the Output

Typical columns:

| Column | Meaning |
|---|---|
| ncalls | Number of calls |
| tottime | Time spent inside function only |
| cumtime | Time including subcalls |
| filename:lineno(function) | Function identifier |

Key insight:
- `tottime` isolates local work,
- `cumtime` reveals full call cost.

### Saving Profile Results

Profiles can be saved for later inspection.

In [ ]:
cProfile.run("main()", "profile_results.prof")

In [ ]:
stats = pstats.Stats("profile_results.prof")

stats.sort_stats("cumtime").print_stats(10)

In [ ]:
!uv pip install line_profiler
%load_ext line_profiler

In [ ]:
def compute():
    total = 0

    for i in range(1_000_000):
        total += np.sqrt(i)

    return total

In [ ]:
%lprun -f compute compute()

## 5. Memory Profiling

Performance is not only about compute.

Memory issues include:
- unnecessary allocations,
- temporary arrays,
- copies,
- fragmentation,
- cache inefficiency.

In [ ]:
!uv pip install memory_profiler
%load_ext memory_profiler

In [ ]:
from memory_profiler import profile

In [ ]:
%%writefile memory_example.py

import numpy as np

def allocate_memory():
    x = np.random.rand(10_000_000)
    y = np.random.rand(10_000_000)

    return x + y

In [ ]:
from memory_example import allocate_memory

%mprun -f allocate_memory allocate_memory()

## Why Bottlenecks Matter

Suppose a simulation code spends:
- 95% of time in one loop,
- 5% elsewhere.

Optimizing everything equally is irrational.

Amdahl’s Law implies that optimization effort should focus on the dominant cost first.

$$
S = \frac{1}{s+(1-s)/N}
$$

Where: 

- $s$: Serial fraction $1-p$
- $N$: Processor count

## Practical Optimization Workflow

A good workflow is:

1. Write correct code
2. Measure runtime
3. Profile bottlenecks
4. Optimize dominant kernels
5. Re-profile
6. Repeat

Never optimize blindly.

## Typical Scientific Python Bottlenecks

| Problem | Typical Solution |
|---|---|
| Python loops | NumPy vectorization |
| Repeated allocations | Buffer reuse |
| Small kernels | Numba |
| Slow Python dispatch | Cython/Pythran |
| Memory bandwidth limits | Blocking/cache locality |
| Serial execution | Parallelization |

## Advanced Profilers

### Snakeviz

Visualize `cProfile` results interactively.

Install:

```bash
pip install snakeviz

snakeviz profile_results.prof
```
### Scalene

```bash
pip install scalene

scalene my_script.py
```


## Final Takeaways

Key lessons:

- Measure before optimizing
- Timing is not profiling
- Repeated measurements matter
- Bottlenecks are usually localized
- Memory performance matters
- Profiling guides optimization strategy

The fastest optimization is:
> removing unnecessary work entirely.